[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Narinder-2006/narinder-flyrank-internship-1/blob/main/work/notebooks/w03_data_contract.ipynb)

# Week 3 — Search Intelligence Data Contract

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring
**Mid-panel month used:** `month=2026-03` (per the assignment's own example — never the
`_sample` table, which is the sealed final month and would leak the outcome window into
anything I "discover" here).

> ⚠️ This notebook needs a Hugging Face **read** token with gated-repo access to
> `FlyRank/internship-warehouse`. On Colab: Settings → Secrets (🔑) → add `HF_TOKEN`.
> Never paste a token directly into a cell — this repo is public.

## 0. Setup (Colab or local)

In [ ]:
%pip -q install duckdb huggingface_hub pandas

In [ ]:
import os, getpass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("connected")

## 1. The contract, in plain words

**1. What one row means for my lane:** one row = one **page, aggregated over one calendar
month** (`client_hash_id` + `content_hash_id` + `month`). The raw grain of
`fact_content_daily_performance` is report_date × client × content — I'm rolling that up to a
monthly grain because Lane 2's decision ("review this page this sprint or not") is a
page-level, not a page-day-level, decision.

**2. Which table(s) I'll use:** `fact_content_daily_performance` (primary — the monthly
aggregates come from here), joined to `dim_content` for content metadata (content type,
intent) and `dim_clients` for the unbalanced-panel check (`gsc_data_start`/`ga4_data_start`).
I am deliberately **not** using `fact_content_query_90d` this week (see point 5).

**3. Which time window:** one mid-panel month, `month=2026-03`, chosen specifically because
it is *not* the sealed final month (`_sample` = June 2026) — using the final month to explore
label logic would let the natural outcome window leak into my "discoveries" before I've even
started modeling.

**4. What I'd predict or rank (label or proxy):** same target family as Week 2 — a
declining-vs-not label for a page, this time built from real daily data instead of the
pre-aggregated starter proxy. This week I'm not finalizing the label, just proving I can build
the features it would need.

**5. One thing I deliberately exclude:** `fact_content_query_90d` entirely, for this
notebook. The data dictionary flags that this table's 90-day window overlaps the most recent
months of the snapshot — if I ever predict an outcome in a recent month, that table's
`impressions_90d` / `*_last30` columns would contain the label period itself. Since I haven't
locked my label window yet, pulling features from a table with a moving, overlapping window is
a leakage risk I'd rather avoid until the label is nailed down. I'm also excluding rows before
each client's `ga4_data_start` from any GA4-based feature — those rows are zero-filled, not
genuinely zero engagement.

## 2. Prove three facts with three queries (`month=2026-03`)

### Query 1 — the grain: one row really is (client, content, date)

In [ ]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_keys
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

grain_check["grain_confirmed"] = grain_check["total_rows"] == grain_check["distinct_keys"]
grain_check

**Read this as:** if `total_rows == distinct_keys`, the raw daily table really is one row per
(client, content, day) with no duplicates — the assumption my monthly rollup in Section 3
depends on.

### Query 2 — my slice's row count and date span

In [ ]:
slice_shape = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

slice_shape

### Query 3 — availability, filtered with `IS TRUE`

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

availability["ga4_available_share"] = (
    availability["ga4_available_rows"] / availability["all_rows"]
).round(3)
availability

**Why this matters:** `ga4_data_available` is zero-filled (not missing) before a client's
`ga4_data_start`. Filtering with `IS TRUE` — instead of just checking `pageviews > 0` — is the
only way to distinguish "genuinely no engagement" from "this client's GA4 history hasn't
started yet." Any GA4-based feature I build has to respect this flag or it silently treats
early-history rows as zero-engagement pages.

## 3. Five features, max — built from `month=2026-03`

Each one gets a one-line "knowable at the decision moment because…" justification. All five
come only from `fact_content_daily_performance` rows dated on or before the decision point
(end of the month), so none of them reach into the future relative to that point.

In [ ]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                   AS impressions_month,
        SUM(gsc_clicks)                                        AS clicks_month,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_month,
        COUNT(*) FILTER (WHERE gsc_impressions > 0)            AS days_with_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_month
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2
""").df()

print(f"{len(feature_frame):,} page-months")
feature_frame.head()

1. **`impressions_month`** — knowable at the decision moment because it's a straight sum of
   GSC impressions already logged for days that have already happened by month-end.
2. **`clicks_month`** — same reasoning: a sum of already-observed daily click counts, nothing
   about days after the decision point.
3. **`avg_position_month`** — knowable because it's the average of daily position readings
   Google Search Console already reported for this month; no future ranking data involved.
4. **`days_with_impressions`** — a count of how many days *within this already-completed
   month* had at least one impression; purely retrospective coverage, not a forecast.
5. **`ctr_month`** — a ratio of two already-observed sums (`clicks_month / impressions_month`);
   derived entirely from data available the moment the month closes, not from anything in the
   following month.

## 4. The trap — spring it, then remove it

**Step A — a quick, honest proxy label** for this exercise: within the same month, did the
second half see materially fewer impressions than the first half? (A crude in-month decline
signal — not the real capstone label, just enough to demonstrate leakage on real data.)

**Step B — quick honest score** using only the five leakage-safe features above.

**Step C — spring the trap**: add `second_half_impressions` itself as a "feature" — the exact
quantity the label is built from — and watch the score jump toward a perfect number.

**Step D — delete it, keep the honest score.**

In [ ]:
halves = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) <= 15) AS first_half_impressions,
        SUM(gsc_impressions) FILTER (WHERE EXTRACT(day FROM report_date) > 15)  AS second_half_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2
""").df()

trap_data = feature_frame.merge(halves, on=["client_hash_id", "content_hash_id"])
trap_data["is_declining_inmonth"] = (
    trap_data["second_half_impressions"] < 0.8 * trap_data["first_half_impressions"]
).astype(int)

print("in-month proxy label rate:", trap_data["is_declining_inmonth"].mean().round(3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_month", "clicks_month", "avg_position_month",
                    "days_with_impressions", "ctr_month"]

model_data = trap_data.dropna(subset=honest_features + ["is_declining_inmonth"])
X = model_data[honest_features]
y = model_data["is_declining_inmonth"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST auc (leakage-safe features only): {honest_auc:.3f}")

In [ ]:
# Step C — spring the trap: add the label-derived column as a "feature"
leaky_features = honest_features + ["second_half_impressions"]

model_data_leak = trap_data.dropna(subset=leaky_features + ["is_declining_inmonth"])
X_leak = model_data_leak[leaky_features]
y_leak = model_data_leak["is_declining_inmonth"]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY auc (second_half_impressions included): {leaky_auc:.3f}")
print(f"jump: {leaky_auc - honest_auc:+.3f}")

**Step D — delete it.** `second_half_impressions` is literally half the quantity the label
(`is_declining_inmonth`) is computed from, so including it isn't the model learning a real
pattern — it's the model reading the answer key. The leaky AUC jumping toward ~1.0 is exactly
that: an artificially perfect score with zero real-world value, because at the actual decision
moment (before the month closes) I would never have `second_half_impressions` available. **The
honest number above — not this one — is the one I keep and report.**

## Named limitation of this slice

One mid-panel month (`2026-03`) cannot capture **seasonality** — content performance can swing
with time of year (e.g. holiday shopping content, back-to-school topics), and a single month's
"declining" pattern here may just be a seasonal dip rather than a genuine structural decline.
Any real label built on top of this month's features should either use a longer training
window across multiple months, or explicitly control for seasonal effects, before I trust its
predictions across the full year.

## 5. Self-check

- [x] Five plain-words contract answers (Section 1)
- [x] Exactly three verification queries with outputs visible: grain, slice shape, availability
      via `IS TRUE` (Section 2)
- [x] Five-feature frame, each with an "available when?" line (Section 3)
- [x] Deliberate-leak experiment shown and then removed, honest score kept (Section 4)
- [x] One named limitation of this slice (seasonality, single-month window)
- [ ] Not yet done: locking the real forward-looking label and window — that's modeling-weeks
      work, not this notebook's job.